# We first try to replicate modulo arith Figure 1 grokking behavior in Jax. 

In [1]:
# 2(c). Sweep over laziness.
# Figure 2. Modular arithmetic.

import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
from flax.training import train_state
import optax
from tqdm import tqdm
import matplotlib.pyplot as plt
from jax import random


In [2]:
p = 23
D = 2 * p  # Input units (each input is a pair of one-hot vectors)
alpha = 0.9  # Fraction of data used for training
scale = 1
seed = 1
N = 100


scales = [1]

class SimpleMLP(nn.Module):
    input_dim: int
    hidden_dim: int
    output_dim: int
    scale: float = 1

    def setup(self):
        self.layer1 = nn.Dense(self.hidden_dim, kernel_init=random.normal, use_bias=False)
        self.layer2 = nn.Dense(self.output_dim, kernel_init=random.normal, use_bias=False)

    def __call__(self, x):
        x = self.layer1(x)
        x = x ** 2  # Quadratic activation
        x = self.layer2(x) * self.scale / (self.input_dim * self.hidden_dim)
        return x * scale

def modulo(x):
    a, b = jnp.where(x == 1)[0]
    b -= p
    m = (a + b) % p
    return jax.nn.one_hot(m, p)

# Create dataset for modular arithmetic task
# Each input is a pair of one-hot vectors; each output is a one-hot vector
X = jnp.array([jnp.concatenate([jax.nn.one_hot(i // p, p), jax.nn.one_hot(i % p, p)]) for i in range(p ** 2)])
y = jnp.array([modulo(x) for x in X])
key = jax.random.PRNGKey(seed)
indices = jax.random.permutation(key, p ** 2)
train_indices = indices[:int(alpha * (p ** 2))]
test_indices = indices[int(alpha * (p ** 2)):]
X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]

In [ ]:
epochs = 10000

trs, tls = [], []
for scale in scales:
    lr = 1e2 / scale ** 2
    tr_losses, te_losses = [], []
    print(f"Grokking modular addition: p: {p}, D: {D}, train-set-frac: {alpha}, scale: {scale}, lr: {round(lr, 2)}, seed: {seed}, N: {N}, epochs: {epochs}")

    model = SimpleMLP(D, N, p, scale)
    params = model.init(key, X_train)['params']

    tx = optax.sgd(learning_rate=lr)
    state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

    @jax.jit
    def loss_fn(params, X, y):
        y_pred = model.apply({'params': params}, X)
        loss = jnp.mean((y_pred - y) ** 2) / float(scale ** 2)
        return loss
        
    grad_fn = jax.value_and_grad(loss_fn)        
    @jax.jit
    def train_step(state, X, y):
        loss, grads = grad_fn(state.params, X, y)
        state = state.apply_gradients(grads=grads)
        return state, loss

    @jax.jit
    def eval_step(params, X, y):
        y_pred = model.apply({'params': params}, X)
        loss = jnp.mean((y_pred - y) ** 2)

        predicted_test = jnp.argmax(y_pred, axis=1)
        correct_test = jnp.sum(predicted_test == jnp.argmax(y, axis=1))

        return loss, correct_test

    # @jax.jit
    def eval_train(params, X, y):
        y_pred_train = model.apply({'params': params}, X)
        predicted_train = jnp.argmax(y_pred_train, axis=1)
        correct_train = jnp.sum(predicted_train == jnp.argmax(y, axis=1))
        return correct_train

    train_loss, test_loss = [], []
    train_acc, test_acc = [], []
    for epoch in tqdm(range(epochs)):
        state, loss = train_step(state, X_train, y_train)
        train_loss.append(loss.item())

        y_pred_train = model.apply({'params': state.params}, X_train)
        predicted_train = jnp.argmax(y_pred_train, axis=1)
        correct_train = jnp.sum(predicted_train == jnp.argmax(y_train, axis=1))
        # train_acc.append(100 * correct_train / len(y_train))
        
        correct_train = eval_train(state.params, X_train, y_train)
        train_acc.append(100 * correct_train / len(y_train))

        test_loss_batch, correct_test = eval_step(state.params, X_test, y_test)
        test_loss.append(test_loss_batch)
        test_acc.append(100 * correct_test / len(y_test))

    trs += [train_acc]
    tls += [test_acc]

    tr_losses += [train_loss]
    te_losses += [test_loss]


colors = ['g', 'b', 'r', 'black']
plt.figure(figsize=(9, 6))
plt.rcParams.update({'font.size': 14})

plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Accuracy', fontsize=20)

for i in range(len(trs)):
    plt.plot(range(epochs), trs[i], color=colors[i], label=rf'Train, $\alpha={scales[i]}$', linestyle='--', linewidth=2.0, dashes=(3, 4))
    plt.plot(range(epochs), tls[i], color=colors[i], label=rf'Test, $\alpha={scales[i]}$', linewidth=2.0)

plt.legend(bbox_to_anchor=(0.67, 0.9), fontsize=16)
plt.tight_layout()
plt.show()

Grokking modular addition: p: 23, D: 46, train-set-frac: 0.9, scale: 1, lr: 100.0, seed: 1, N: 100, epochs: 10000


  0%|▏                                                                                                    | 16/10000 [00:01<09:54, 16.78it/s]